# 03 — GraphSAGE Training

**SONATAM** — *Symbolic Ontology & Neural Audio Transcription Architecture for Music*

This notebook trains the **heterogeneous GraphSAGE** model for
**link prediction** on the music knowledge graph.

Pipeline:
1. Load the HeteroData (from NB 02)
2. Configure & train the GraphSAGE link-prediction model
3. Evaluate: AUC-ROC, Average Precision, MRR, Hits@K
4. Visualise training curves + embedding t-SNE

---

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────
from pathlib import Path

import torch

from sonata.config.settings import CFG
from sonata.notebook_utils import rp, pp  # pp() = absolute path from config string
from sonata.models import (
    GraphSAGELinkPredModel,
    LinkPredTrainer,
    TrainerConfig,
    evaluate_link_prediction,
    plot_training_curves,
    plot_embedding_tsne,
    plot_score_distribution,
)

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Load HeteroData

In [ ]:
data_path = pp("data/processed/hetero_data.pt")
print(f"Loading: {rp(data_path)}")

data = torch.load(data_path, weights_only=False)
print(data)
print(f"\nNode types: {data.node_types}")
print(f"Edge types: {data.edge_types}")

## 2. Configure training

In [ ]:
config = TrainerConfig.from_config()
print("TrainerConfig:")
for k, v in config.__dict__.items():
    print(f"  {k}: {v}")

## 3. Train the GraphSAGE model

In [ ]:
trainer = LinkPredTrainer(config)
model, history = trainer.fit(data, verbose=True)

## 4. Training curves

In [ ]:
plot_training_curves(history, save_path=str(pp("data/processed/training_curves.png")))

## 5. Evaluation on test split

In [ ]:
from sonata.kg.converter import HeteroGraphConverter

converter = HeteroGraphConverter()
splits = converter.create_link_split(
    data,
    edge_type=config.target_edge_type,
    val_ratio=config.val_ratio,
    test_ratio=config.test_ratio,
    seed=config.seed,
)

test_data = splits["test"]
metrics = evaluate_link_prediction(
    model, test_data, config.target_edge_type, device=config.device,
)

print("\n═══ Test Metrics ═══")
for k, v in metrics.items():
    print(f"  {k:>12s}: {v:.4f}")

## 6. Embedding visualisation

In [ ]:
# Get embeddings from the encoder
model.eval()
with torch.no_grad():
    z_dict = model.encode(data.x_dict, data.edge_index_dict)

if "MusicalPiece" in z_dict:
    plot_embedding_tsne(
        z_dict["MusicalPiece"],
        node_type="MusicalPiece",
        save_path=str(pp("data/processed/tsne_embeddings.png")),
    )

## 7. Score distribution

In [ ]:
# Predict scores on test edges
et = config.target_edge_type
src_type, _, dst_type = et

with torch.no_grad():
    test_data_d = test_data.to(config.device)
    pred = model(
        test_data_d.x_dict, test_data_d.edge_index_dict,
        test_data_d[et].edge_label_index, src_type, dst_type,
    )
    scores = torch.sigmoid(pred)
    labels = test_data_d[et].edge_label

plot_score_distribution(
    scores, labels,
    save_path=str(pp("data/processed/score_distribution.png")),
)